# Empirical Bayes Contextual Offsets

Stage 2 of the hierarchical modeling track. This notebook tests small, shrinkage-controlled residual corrections on top of the current Huber + additive Hilbert residual model.

## 1. Setup

In [1]:
"Configure imports, paths, and empirical-Bayes priors."

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__hierarchical_empirical_bayes_offsets"

ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
WELL_AUDIT_PATH = ARTIFACT_TABLES / "model__hierarchical_residual_bias_well_summary__2026-06-22.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514

OFFSET_CLIP = 12.0
MIN_WELL_WEIGHT = 3

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__hierarchical_empirical_bayes_offsets


## 2. Modeling Assumptions

The empirical-Bayes prior is conservative: well-level corrections are centered at zero, learned only from other folds, clipped to avoid large extrapolations, and blended with shrinkage factors. This is a practical partial-pooling approximation before moving to a full Bayesian hierarchy.

In [2]:
"Load the current best residual sample and the Stage 1 well-level descriptors."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "y_true", "y_pred", "residual", "eval_progress", "distance_from_boundary_md", "abs_projected_delta_10"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["distance_log1p"] = np.log1p(residuals["distance_from_boundary_md"].clip(lower=0))
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]

well_audit = pd.read_csv(WELL_AUDIT_PATH)
well_audit["fold"] = pd.to_numeric(well_audit["fold"], errors="coerce").astype(int)

print(f"rows={len(residuals):,}")
print(f"wells={residuals['wellname'].nunique():,}")
print(f"folds={sorted(residuals['fold'].dropna().unique().astype(int).tolist())}")

rows=5,400
wells=773
folds=[1, 2, 3]


## 3. Offset Feature Sets And Candidate Arms

In [3]:
"Define descriptor features, shrinkage arms, and scoring helpers."

CORE_DESCRIPTOR_COLS = [
    "max_distance_from_boundary_md",
    "max_distance_log1p",
    "mean_distance_log1p",
    "xy_span",
    "md_span",
    "source_rows",
    "known_tvt_fraction",
    "known_tvt_rows",
    "gr_missing_rate",
    "gr_mean",
    "gr_std",
    "recent_tvt_slope_last_50",
    "mean_abs_projected_delta_10",
    "max_abs_projected_delta_10",
]

ROW_COORD_COLS = [
    "eval_progress",
    "distance_log1p",
    "abs_projected_delta_10",
]

CANDIDATES = [
    {"name": "baseline_no_offset", "kind": "baseline", "shrink": 0.0, "alpha": 0.0, "prior": "zero correction"},
    {"name": "global_bias_shrink_025", "kind": "global", "shrink": 0.25, "alpha": 0.0, "prior": "global residual mean with 75 percent shrinkage to zero"},
    {"name": "global_bias_shrink_050", "kind": "global", "shrink": 0.50, "alpha": 0.0, "prior": "global residual mean with 50 percent shrinkage to zero"},
    {"name": "well_intercept_ridge_a10_s025", "kind": "well_intercept", "shrink": 0.25, "alpha": 10.0, "prior": "ridge descriptor intercept with strong zero shrinkage"},
    {"name": "well_intercept_ridge_a100_s025", "kind": "well_intercept", "shrink": 0.25, "alpha": 100.0, "prior": "ridge descriptor intercept with stronger coefficient shrinkage"},
    {"name": "well_intercept_ridge_a100_s050", "kind": "well_intercept", "shrink": 0.50, "alpha": 100.0, "prior": "ridge descriptor intercept with moderate zero shrinkage"},
    {"name": "well_intercept_huber_s025", "kind": "well_intercept_huber", "shrink": 0.25, "alpha": 0.01, "prior": "robust descriptor intercept with strong zero shrinkage"},
    {"name": "well_intercept_progress_slope_a100_s025", "kind": "intercept_progress", "shrink": 0.25, "alpha": 100.0, "prior": "descriptor intercept plus progress slope, strongly shrunk"},
    {"name": "row_context_ridge_a100_s025", "kind": "row_context", "shrink": 0.25, "alpha": 100.0, "prior": "row-level contextual residual with strong zero shrinkage"},
]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def clipped(values):
    return np.clip(np.asarray(values, dtype=float), -OFFSET_CLIP, OFFSET_CLIP)

def descriptor_pipeline(alpha):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            ("model", Ridge(alpha=alpha)),
        ]
    )

def huber_pipeline(alpha):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
            ("model", HuberRegressor(alpha=alpha, epsilon=1.35, max_iter=500)),
        ]
    )

def row_pipeline(alpha):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha)),
        ]
    )

print(pd.DataFrame(CANDIDATES)[["name", "kind", "shrink", "alpha", "prior"]])

                                      name                  kind  shrink  \
0                       baseline_no_offset              baseline    0.00   
1                   global_bias_shrink_025                global    0.25   
2                   global_bias_shrink_050                global    0.50   
3            well_intercept_ridge_a10_s025        well_intercept    0.25   
4           well_intercept_ridge_a100_s025        well_intercept    0.25   
5           well_intercept_ridge_a100_s050        well_intercept    0.50   
6                well_intercept_huber_s025  well_intercept_huber    0.25   
7  well_intercept_progress_slope_a100_s025    intercept_progress    0.25   
8              row_context_ridge_a100_s025           row_context    0.25   

    alpha                                              prior  
0    0.00                                    zero correction  
1    0.00  global residual mean with 75 percent shrinkage...  
2    0.00  global residual mean with 50 percent sh

## 4. Cross-Fold Offset Evaluation

In [4]:
"Evaluate each offset arm with fold-held-out residuals and no same-fold offset training."

def well_slope_targets(train_rows):
    rows = []
    for wellname, group in train_rows.groupby("wellname"):
        if len(group) < 4 or group["eval_progress"].nunique() < 2:
            intercept = float(group["residual"].mean())
            slope = 0.0
        else:
            x = group["eval_progress"].to_numpy(dtype=float) - group["eval_progress"].mean()
            y = group["residual"].to_numpy(dtype=float)
            slope = float(np.polyfit(x, y, deg=1)[0])
            intercept = float(y.mean())
        rows.append({"wellname": wellname, "residual_intercept": intercept, "residual_progress_slope": slope})
    return pd.DataFrame(rows)

predicted_rows = []
fold_metric_rows = []

for candidate in CANDIDATES:
    for fold in sorted(residuals["fold"].dropna().unique()):
        train_rows = residuals.loc[~residuals["fold"].eq(fold)].copy()
        valid_rows = residuals.loc[residuals["fold"].eq(fold)].copy()
        train_wells = well_audit.loc[~well_audit["fold"].eq(fold)].copy()
        valid_wells = well_audit.loc[well_audit["fold"].eq(fold)].copy()
        valid_well_features = valid_rows[["wellname"]].merge(valid_wells, on="wellname", how="left")
        correction = np.zeros(len(valid_rows), dtype=float)
        raw_correction = np.zeros(len(valid_rows), dtype=float)
        if candidate["kind"] == "global":
            raw_value = float(train_rows["residual"].mean())
            raw_correction = np.repeat(raw_value, len(valid_rows))
            correction = candidate["shrink"] * raw_correction
        elif candidate["kind"] in ["well_intercept", "well_intercept_huber"]:
            train_model_frame = train_wells.copy()
            train_model_frame["sample_weight"] = np.sqrt(train_model_frame["n_residual_rows"].clip(lower=MIN_WELL_WEIGHT))
            model = descriptor_pipeline(candidate["alpha"]) if candidate["kind"] == "well_intercept" else huber_pipeline(candidate["alpha"])
            fit_kwargs = {}
            if candidate["kind"] == "well_intercept":
                fit_kwargs["model__sample_weight"] = train_model_frame["sample_weight"].to_numpy(dtype=float)
            model.fit(train_model_frame[CORE_DESCRIPTOR_COLS], train_model_frame["mean_residual"], **fit_kwargs)
            raw_correction = model.predict(valid_well_features[CORE_DESCRIPTOR_COLS])
            correction = candidate["shrink"] * clipped(raw_correction)
        elif candidate["kind"] == "intercept_progress":
            slope_target = well_slope_targets(train_rows)
            train_model_frame = train_wells.merge(slope_target, on="wellname", how="inner")
            intercept_model = descriptor_pipeline(candidate["alpha"])
            slope_model = descriptor_pipeline(candidate["alpha"] * 2.0)
            weights = np.sqrt(train_model_frame["n_residual_rows"].clip(lower=MIN_WELL_WEIGHT)).to_numpy(dtype=float)
            intercept_model.fit(train_model_frame[CORE_DESCRIPTOR_COLS], train_model_frame["residual_intercept"], model__sample_weight=weights)
            slope_model.fit(train_model_frame[CORE_DESCRIPTOR_COLS], train_model_frame["residual_progress_slope"], model__sample_weight=weights)
            intercept_raw = intercept_model.predict(valid_well_features[CORE_DESCRIPTOR_COLS])
            slope_raw = slope_model.predict(valid_well_features[CORE_DESCRIPTOR_COLS])
            progress_centered = valid_rows["eval_progress"].to_numpy(dtype=float) - valid_well_features["mean_eval_progress"].fillna(0.5).to_numpy(dtype=float)
            raw_correction = intercept_raw + slope_raw * progress_centered
            correction = candidate["shrink"] * clipped(raw_correction)
        elif candidate["kind"] == "row_context":
            train_row_features = train_rows.merge(well_audit, on=["fold", "wellname"], how="left")
            valid_row_features = valid_rows.merge(well_audit, on=["fold", "wellname"], how="left")
            row_features = CORE_DESCRIPTOR_COLS + ROW_COORD_COLS
            model = row_pipeline(candidate["alpha"])
            model.fit(train_row_features[row_features], train_row_features["residual"])
            raw_correction = model.predict(valid_row_features[row_features])
            correction = candidate["shrink"] * clipped(raw_correction)
        y_pred_offset = valid_rows["baseline_pred"].to_numpy(dtype=float) + correction
        eval_frame = valid_rows[["fold", "wellname", "row_idx", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "distance_log1p", "abs_projected_delta_10"]].copy()
        eval_frame["candidate"] = candidate["name"]
        eval_frame["raw_offset"] = raw_correction
        eval_frame["offset"] = correction
        eval_frame["y_pred"] = y_pred_offset
        eval_frame["residual_after_offset"] = eval_frame["y_true"] - eval_frame["y_pred"]
        predicted_rows.append(eval_frame)
        fold_metric_rows.append(
            {
                "candidate": candidate["name"],
                "fold": int(fold),
                "kind": candidate["kind"],
                "shrink": candidate["shrink"],
                "alpha": candidate["alpha"],
                "prior": candidate["prior"],
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "rmse": rmse(valid_rows["y_true"], y_pred_offset),
                "mae": float(mean_absolute_error(valid_rows["y_true"], y_pred_offset)),
                "mean_offset": float(np.mean(correction)),
                "offset_std": float(np.std(correction)),
                "mean_residual_after_offset": float(np.mean(eval_frame["residual_after_offset"])),
            }
        )

predictions = pd.concat(predicted_rows, ignore_index=True)
fold_metrics = pd.DataFrame(fold_metric_rows)
summary = (
    fold_metrics.groupby(["candidate", "kind", "shrink", "alpha", "prior"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_abs_fold_bias=("mean_residual_after_offset", lambda s: float(np.mean(np.abs(s)))),
        mean_offset_abs=("mean_offset", lambda s: float(np.mean(np.abs(s)))),
        offset_std_mean=("offset_std", "mean"),
    )
)
sample_baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_offset"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - sample_baseline_rmse
summary["rmse_delta_vs_full_reference"] = summary["rmse_mean"] - LOCAL_REFERENCE_RMSE
summary = summary.sort_values("rmse_mean")
summary

,candidate,kind,shrink,alpha,prior,rmse_mean,rmse_std,mae_mean,mean_abs_fold_bias,mean_offset_abs,offset_std_mean,rmse_delta_vs_sample_baseline,rmse_delta_vs_full_reference
2,global_bias_shrink_050,global,0.50,0.00,global residual mean with 50 percent shrinkage...,14.737753,0.484286,10.418131,0.311759,0.134383,1.850372e-17,-0.000792,0.379153
1,global_bias_shrink_025,global,0.25,0.00,global residual mean with 75 percent shrinkage...,14.737964,0.485861,10.415375,0.307460,0.067192,9.251859e-18,-0.000581,0.379364
5,well_intercept_progress_slope_a100_s025,intercept_progress,0.25,100.00,"descriptor intercept plus progress slope, stro...",14.738049,0.480755,10.420319,0.306529,0.068375,4.240561e-01,-0.000496,0.379449
0,baseline_no_offset,baseline,0.00,0.00,zero correction,14.738545,0.487053,10.413072,0.303161,0.000000,0.000000e+00,0.000000,0.379945
6,well_intercept_ridge_a100_s025,well_intercept,0.25,100.00,ridge descriptor intercept with stronger coeff...,14.745428,0.487903,10.426823,0.306969,0.069112,3.322593e-01,0.006883,0.386828
3,row_context_ridge_a100_s025,row_context,0.25,100.00,row-level contextual residual with strong zero...,14.745828,0.485188,10.428170,0.292917,0.083162,3.781970e-01,0.007283,0.387228
8,well_intercept_ridge_a10_s025,well_intercept,0.25,10.00,ridge descriptor intercept with strong zero sh...,14.746283,0.484277,10.426060,0.284198,0.097318,3.726492e-01,0.007738,0.387683
4,well_intercept_huber_s025,well_intercept_huber,0.25,0.01,robust descriptor intercept with strong zero s...,14.755351,0.487981,10.429105,0.282112,0.115011,4.093980e-01,0.016806,0.396751
7,well_intercept_ridge_a100_s050,well_intercept,0.50,100.00,ridge descriptor intercept with moderate zero ...,14.760480,0.485497,10.449281,0.310777,0.138225,6.645186e-01,0.021935,0.401880


## 5. Well-Level Diagnostics

In [5]:
"Summarize best-candidate residual behavior by well for the next hierarchy decision."

best_candidate = summary.iloc[0]["candidate"]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
well_after = (
    best_predictions.groupby(["fold", "wellname"], as_index=False)
    .agg(
        n_rows=("residual_after_offset", "size"),
        mean_residual_after_offset=("residual_after_offset", "mean"),
        mae_after_offset=("residual_after_offset", lambda s: float(np.mean(np.abs(s)))),
        rmse_after_offset=("residual_after_offset", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_offset=("offset", "mean"),
        offset_std=("offset", "std"),
    )
)
well_before = (
    residuals.groupby(["fold", "wellname"], as_index=False)
    .agg(
        mean_residual_before=("baseline_residual", "mean"),
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
    )
)
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_mean_residual_before"] = well_diagnostics["mean_residual_before"].abs()
well_diagnostics["abs_mean_residual_after"] = well_diagnostics["mean_residual_after_offset"].abs()
well_diagnostics["abs_bias_delta"] = well_diagnostics["abs_mean_residual_after"] - well_diagnostics["abs_mean_residual_before"]
well_diagnostics = well_diagnostics.sort_values("abs_bias_delta")

findings = []
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_offset")].iloc[0]
findings.append(
    {
        "finding": "best_candidate",
        "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs baseline sample RMSE {baseline_row['rmse_mean']:.4f}",
        "modeling_implication": "Promote only if improvement is meaningful and fold-stable.",
    }
)
findings.append(
    {
        "finding": "sample_baseline_delta",
        "value": f"best candidate delta vs sample no-offset baseline {best_row['rmse_delta_vs_sample_baseline']:.4f}",
        "modeling_implication": "Negative means the offset improves the same sampled residual frame.",
    }
)
findings.append(
    {
        "finding": "best_bias_change",
        "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}",
        "modeling_implication": "Offset should reduce well bias without worsening row-level RMSE.",
    }
)
findings = pd.DataFrame(findings)

print(f"best_candidate={best_candidate}")
print(summary.head(10).to_string(index=False))

best_candidate=global_bias_shrink_050
                              candidate                 kind  shrink  alpha                                                          prior  rmse_mean  rmse_std  mae_mean  mean_abs_fold_bias  mean_offset_abs  offset_std_mean  rmse_delta_vs_sample_baseline  rmse_delta_vs_full_reference
                 global_bias_shrink_050               global    0.50   0.00         global residual mean with 50 percent shrinkage to zero  14.737753  0.484286 10.418131            0.311759         0.134383     1.850372e-17                      -0.000792                      0.379153
                 global_bias_shrink_025               global    0.25   0.00         global residual mean with 75 percent shrinkage to zero  14.737964  0.485861 10.415375            0.307460         0.067192     9.251859e-18                      -0.000581                      0.379364
well_intercept_progress_slope_a100_s025   intercept_progress    0.25 100.00      descriptor intercept plus 

## 6. Export Artifacts

In [6]:
"Export empirical-Bayes offset metrics, diagnostics, report, and run bundle."

fold_metrics_path = ARTIFACT_TABLES / f"model__hierarchical_empirical_bayes_offsets_fold_metrics__{DATE_STAMP}.csv"
summary_path = ARTIFACT_TABLES / f"model__hierarchical_empirical_bayes_offsets_summary__{DATE_STAMP}.csv"
residual_sample_path = ARTIFACT_TABLES / f"model__hierarchical_empirical_bayes_offsets_residual_sample__{DATE_STAMP}.csv"
well_diagnostics_path = ARTIFACT_TABLES / f"model__hierarchical_empirical_bayes_offsets_well_diagnostics__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__hierarchical_empirical_bayes_offsets_findings__{DATE_STAMP}.csv"
comparison_path = ARTIFACT_TABLES / f"model__hierarchical_empirical_bayes_offsets_comparison__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__hierarchical_empirical_bayes_offsets__{DATE_STAMP}.md"

fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
predictions.loc[predictions["candidate"].eq(best_candidate)].to_csv(residual_sample_path, index=False)
well_diagnostics.to_csv(well_diagnostics_path, index=False)
findings.to_csv(findings_path, index=False)

comparison = pd.DataFrame(
    [
        {
            "date": DATE_STAMP,
            "model": best_candidate,
            "local_rmse": float(best_row["rmse_mean"]),
            "local_rmse_std": float(best_row["rmse_std"]),
            "delta_vs_sample_baseline": float(best_row["rmse_delta_vs_sample_baseline"]),
            "delta_vs_full_huber_hilbert_reference": float(best_row["rmse_delta_vs_full_reference"]),
            "public_lb_reference": PUBLIC_REFERENCE_RMSE,
            "decision": "promote_to_contextual_huber" if best_row["rmse_delta_vs_sample_baseline"] <= -0.05 else "do_not_promote_yet",
            "notes": "Empirical-Bayes contextual offset on sampled out-of-fold residuals; no Kaggle submission.",
        }
    ]
)
comparison.to_csv(comparison_path, index=False)

def markdown_table(frame):
    display_frame = frame.copy()
    for col in display_frame.select_dtypes(include=["float"]).columns:
        display_frame[col] = display_frame[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
    columns = list(display_frame.columns)
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in display_frame.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, divider] + rows)

report_lines = [
    "# Hierarchical Empirical Bayes Offsets",
    "",
    f"Date: {DATE_STAMP}",
    "",
    "## Purpose",
    "",
    "Test conservative contextual residual offsets on top of the current Huber + additive Hilbert residual model.",
    "",
    "## Priors And Assumptions",
    "",
    f"- Corrections are residual offsets centered on zero and clipped to +/- {OFFSET_CLIP:.1f} TVT units before shrinkage.",
    "- Candidate shrinkage factors are fixed before scoring and evaluated out of fold.",
    "- Well-level effects are predicted from test-safe descriptors, not raw well IDs.",
    "- This is empirical Bayes / partial-pooling screening, not a full posterior model.",
    "",
    "## Results",
    "",
    markdown_table(summary.head(10)),
    "",
    "## Findings",
    "",
    markdown_table(findings),
    "",
    "## Decision",
    "",
    f"Best candidate `{best_candidate}` scored **{best_row['rmse_mean']:.4f}** RMSE, delta **{best_row['rmse_delta_vs_sample_baseline']:.4f}** vs the sample no-offset baseline and **{best_row['rmse_delta_vs_full_reference']:.4f}** vs the stored full Huber + Hilbert reference. Use the promotion gate in `context/STATUS.md` before moving to contextual Huber hierarchy.",
    "",
    "## Artifacts",
    "",
    f"- Fold metrics: `{fold_metrics_path}`",
    f"- Summary: `{summary_path}`",
    f"- Residual sample: `{residual_sample_path}`",
    f"- Well diagnostics: `{well_diagnostics_path}`",
    f"- Findings: `{findings_path}`",
    f"- Comparison row: `{comparison_path}`",
]
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

for path in [fold_metrics_path, summary_path, residual_sample_path, well_diagnostics_path, findings_path, comparison_path, report_path]:
    shutil.copy2(path, ARTIFACT_LOGS / path.name)
metadata = {
    "run_id": RUN_ID,
    "best_candidate": str(best_candidate),
    "best_rmse": float(best_row["rmse_mean"]),
    "best_delta_vs_sample_baseline": float(best_row["rmse_delta_vs_sample_baseline"]),
    "best_delta_vs_full_reference": float(best_row["rmse_delta_vs_full_reference"]),
    "reference_rmse": LOCAL_REFERENCE_RMSE,
    "public_reference_rmse": PUBLIC_REFERENCE_RMSE,
    "decision": str(comparison.iloc[0]["decision"]),
}
(ARTIFACT_LOGS / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(report_path)
print(summary_path)
print(fold_metrics_path)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__hierarchical_empirical_bayes_offsets__2026-06-23.md
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_empirical_bayes_offsets_summary__2026-06-23.csv
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_empirical_bayes_offsets_fold_metrics__2026-06-23.csv


## Results

This notebook is a practical hierarchy screen. It decides whether simple contextual offsets are strong enough to justify folding contextual hierarchy into the Huber mean model.